# Gold Mart Validation

Validates the Gold dimensions and fact table created by `04_gold_mart_creation.ipynb`.

This notebook is read-only: it does not create, replace, update, or delete Gold tables.

## Validation Scope and Pass Standard

The execution order is Gold creation followed by Gold validation. This notebook verifies:

- dimension keys are unique and non-null according to their documented policy
- Silver and Gold fact row counts and retained measures reconcile
- fact-row keys are unique and non-null
- every fact foreign key matches a dimension member
- dimension joins preserve the fact-table row count
- unchanged Silver inputs produce an unchanged Gold validation signature

In [0]:
SHOW TABLES IN `ftw-week-08`.`03_gold`;

## Validate DIM_DATE

Check continuous coverage, unique key mapping, calendar attributes, and retained out-of-analysis-window dates.

In [0]:
SELECT
    COUNT(*) AS row_count,
    COUNT(DISTINCT date_key) AS unique_date_keys,
    MIN(full_date) AS minimum_date,
    MAX(full_date) AS maximum_date,

    DATEDIFF(MAX(full_date), MIN(full_date)) + 1
        AS expected_row_count,

    COUNT(*) - (
        DATEDIFF(MAX(full_date), MIN(full_date)) + 1
    ) AS missing_date_count,

    SUM(
        CASE
            WHEN date_key != CAST(DATE_FORMAT(full_date, 'yyyyMMdd') AS INT)
            THEN 1
            ELSE 0
        END
    ) AS invalid_date_key_count,

    SUM(
        CASE
            WHEN year != YEAR(full_date)
              OR quarter != QUARTER(full_date)
              OR month_number != MONTH(full_date)
              OR day_of_month != DAYOFMONTH(full_date)
              OR day_of_week != DAYOFWEEK(full_date)
              OR is_weekend != (DAYOFWEEK(full_date) IN (1, 7))
            THEN 1
            ELSE 0
        END
    ) AS invalid_date_attribute_count,

    SUM(
        CASE
            WHEN is_holiday IS NOT NULL THEN 1
            ELSE 0
        END
    ) AS unexpected_holiday_value_count

FROM `ftw-week-08`.`03_gold`.dim_date;

In [0]:
SELECT
    COUNT(*) AS affected_trip_rows,
    MIN(lpep_pickup_datetime) AS earliest_pickup_datetime,
    MAX(lpep_pickup_datetime) AS latest_pickup_datetime,
    MIN(lpep_dropoff_datetime) AS earliest_dropoff_datetime,
    MAX(lpep_dropoff_datetime) AS latest_dropoff_datetime
FROM `ftw-week-08`.`02_silver`.green_taxi
WHERE TO_DATE(lpep_pickup_datetime) = DATE '2008-12-31'
   OR TO_DATE(lpep_dropoff_datetime) = DATE '2008-12-31';

In [0]:
SELECT *
FROM `ftw-week-08`.`02_silver`.green_taxi
WHERE TO_DATE(lpep_pickup_datetime) = DATE '2008-12-31'
   OR TO_DATE(lpep_dropoff_datetime) = DATE '2008-12-31';

In [0]:
SELECT *
FROM `ftw-week-08`.`03_gold`.dim_date
ORDER BY full_date
LIMIT 10;

In [0]:
DESCRIBE TABLE `ftw-week-08`.`03_gold`.dim_date;

## DIM_DATE Verified Conditions

- row count equals the unique date-key count
- row count equals the continuous expected date count
- no date is missing between the minimum and maximum retained dates
- pickup and drop-off dates resolve to the dimension

## Validate DIM_TIME

The dimension contains 25 unique members: key `0` for Unknown and keys `1–24` for hours `00:00–23:00`. The validation checks the Unknown-member definition and every hour-to-key mapping.

In [0]:
SELECT
    COUNT(*) AS row_count,
    COUNT(DISTINCT time_key) AS unique_time_keys,
    MIN(time_key) AS minimum_time_key,
    MAX(time_key) AS maximum_time_key,

    SUM(
        CASE
            WHEN time_key = 0
             AND hour_24 IS NULL
             AND hour_label = 'Unknown'
             AND day_period = 'Unknown'
            THEN 0
            WHEN time_key = 0 THEN 1
            ELSE 0
        END
    ) AS invalid_unknown_row_count,

    SUM(
        CASE
            WHEN time_key BETWEEN 1 AND 24
             AND hour_24 = time_key - 1
            THEN 0
            WHEN time_key BETWEEN 1 AND 24 THEN 1
            ELSE 0
        END
    ) AS invalid_hour_mapping_count

FROM `ftw-week-08`.`03_gold`.dim_time;

## Validate DIM_TAXI_ZONE

Check one unique and non-null key per trusted Silver `LocationID`, with no key/reference mismatch.

In [0]:
SELECT
    COUNT(*) AS row_count,
    COUNT(DISTINCT taxi_zone_key) AS unique_taxi_zone_keys,

    COUNT(*) - COUNT(DISTINCT taxi_zone_key)
        AS duplicate_taxi_zone_key_count,

    SUM(
        CASE
            WHEN taxi_zone_key IS NULL THEN 1
            ELSE 0
        END
    ) AS null_taxi_zone_key_count,

    SUM(
        CASE
            WHEN location_id IS NULL THEN 1
            ELSE 0
        END
    ) AS null_location_id_count,

    SUM(
        CASE
            WHEN taxi_zone_key != location_id THEN 1
            ELSE 0
        END
    ) AS key_reference_mismatch_count

FROM `ftw-week-08`.`03_gold`.dim_taxi_zone;

## DIM_TAXI_ZONE Verified Conditions

The validation checks one Gold row per trusted Silver `LocationID`, unique and non-null `taxi_zone_key` values, and zero key/reference mismatches.

## Validate DIM_WEATHER_HOUR

Compare accepted Silver Weather hours with actual Gold Weather members, validate key/timestamp uniqueness and key derivation, and confirm exactly one separate Unknown member.

In [0]:
WITH source_summary AS (
    SELECT
        COUNT(*) AS source_row_count,
        COUNT(DISTINCT weather_datetime) AS source_unique_weather_hours
    FROM `ftw-week-08`.`02_silver`.weather
),
gold_actual_weather AS (
    SELECT
        COUNT(*) AS gold_actual_row_count,
        COUNT(DISTINCT weather_hour_key) AS unique_weather_hour_keys,
        COUNT(DISTINCT weather_timestamp_local) AS unique_weather_timestamps,
        COUNT_IF(weather_hour_key IS NULL) AS null_weather_hour_key_count,
        COUNT_IF(weather_timestamp_local IS NULL) AS null_weather_timestamp_count,
        COUNT_IF(
            weather_hour_key != CAST(
                DATE_FORMAT(weather_timestamp_local, 'yyyyMMddHH') AS BIGINT
            )
        ) AS invalid_weather_key_mapping_count
    FROM `ftw-week-08`.`03_gold`.dim_weather_hour
    WHERE weather_hour_key != 0
),
unknown_member AS (
    SELECT COUNT(*) AS unknown_member_count
    FROM `ftw-week-08`.`03_gold`.dim_weather_hour
    WHERE weather_hour_key = 0
)
SELECT
    s.source_row_count,
    s.source_unique_weather_hours,
    g.gold_actual_row_count,
    g.unique_weather_hour_keys,
    g.unique_weather_timestamps,
    g.gold_actual_row_count - s.source_row_count AS actual_row_count_difference,
    g.null_weather_hour_key_count,
    g.null_weather_timestamp_count,
    g.invalid_weather_key_mapping_count,
    u.unknown_member_count
FROM source_summary AS s
CROSS JOIN gold_actual_weather AS g
CROSS JOIN unknown_member AS u;

In [0]:
SELECT *
FROM `ftw-week-08`.`03_gold`.dim_weather_hour
ORDER BY weather_hour_key
LIMIT 10;

## DIM_WEATHER_HOUR Verified Conditions

- actual Gold Weather rows equal accepted Silver Weather rows
- actual keys and local timestamps are unique and non-null
- every actual key matches `yyyyMMddHH` of its local timestamp
- one separate Unknown member exists at `weather_hour_key = 0`
- no Gold-side Weather deduplication is applied

## Reconcile FACT_GREEN_TAXI_TRIP with Silver

The fact must retain every accepted Silver row. Row-count, trip-count, and retained-measure differences must be zero after the documented rounding tolerance.

In [0]:
WITH silver_summary AS (
    SELECT
        COUNT(*) AS row_count,
        SUM(trip_distance) AS trip_distance,
        SUM(trip_duration_minutes) AS trip_duration_minutes,
        SUM(fare_amount) AS fare_amount,
        SUM(extra) AS extra,
        SUM(mta_tax) AS mta_tax,
        SUM(tip_amount) AS tip_amount,
        SUM(tolls_amount) AS tolls_amount,
        SUM(improvement_surcharge) AS improvement_surcharge,
        SUM(congestion_surcharge) AS congestion_surcharge,
        SUM(cbd_congestion_fee) AS cbd_congestion_fee,
        SUM(total_amount) AS total_amount
    FROM `ftw-week-08`.`02_silver`.green_taxi
),
gold_summary AS (
    SELECT
        COUNT(*) AS row_count,
        COUNT(DISTINCT trip_key) AS unique_trip_keys,
        COUNT_IF(trip_key IS NULL) AS null_trip_keys,
        SUM(trip_count) AS trip_count,
        SUM(trip_distance) AS trip_distance,
        SUM(trip_duration_minutes) AS trip_duration_minutes,
        SUM(fare_amount) AS fare_amount,
        SUM(extra) AS extra,
        SUM(mta_tax) AS mta_tax,
        SUM(tip_amount) AS tip_amount,
        SUM(tolls_amount) AS tolls_amount,
        SUM(improvement_surcharge) AS improvement_surcharge,
        SUM(congestion_surcharge) AS congestion_surcharge,
        SUM(cbd_congestion_fee) AS cbd_congestion_fee,
        SUM(total_amount) AS total_amount
    FROM `ftw-week-08`.`03_gold`.fact_green_taxi_trip
)
SELECT
    s.row_count AS silver_rows,
    g.row_count AS gold_rows,
    g.row_count - s.row_count AS row_difference,
    g.unique_trip_keys,
    g.row_count - g.unique_trip_keys AS duplicate_trip_key_count,
    g.null_trip_keys,
    g.trip_count - s.row_count AS trip_count_difference,
    ROUND(g.trip_distance - s.trip_distance, 6) AS distance_difference,
    ROUND(g.trip_duration_minutes - s.trip_duration_minutes, 6) AS duration_difference,
    ROUND(g.fare_amount - s.fare_amount, 6) AS fare_difference,
    ROUND(g.extra - s.extra, 6) AS extra_difference,
    ROUND(g.mta_tax - s.mta_tax, 6) AS mta_tax_difference,
    ROUND(g.tip_amount - s.tip_amount, 6) AS tip_difference,
    ROUND(g.tolls_amount - s.tolls_amount, 6) AS tolls_difference,
    ROUND(
        g.improvement_surcharge - s.improvement_surcharge, 6
    ) AS improvement_surcharge_difference,
    ROUND(
        g.congestion_surcharge - s.congestion_surcharge, 6
    ) AS congestion_surcharge_difference,
    ROUND(
        g.cbd_congestion_fee - s.cbd_congestion_fee, 6
    ) AS cbd_congestion_fee_difference,
    ROUND(g.total_amount - s.total_amount, 6) AS total_amount_difference
FROM silver_summary AS s
CROSS JOIN gold_summary AS g;

## FACT_GREEN_TAXI_TRIP Acceptance Criteria

- Silver and Gold row counts match
- `SUM(trip_count)` equals the Gold fact-row count
- fact-row keys are unique and non-null
- all retained measure differences are zero after the documented rounding tolerance
- no accepted Silver row is removed by Gold-side deduplication

Use the executed reconciliation output as the evidence; do not rely on hard-coded counts in Markdown.

## Validate Fact Grain and Technical Row Key

Confirm one non-null and unique technical fact-row key for every accepted source record. The fingerprint is not presented as a proven real-world trip business key.

In [0]:
SELECT
    COUNT(*) AS row_count,
    COUNT(DISTINCT trip_key) AS unique_trip_keys,
    COUNT(*) - COUNT(DISTINCT trip_key) AS duplicate_trip_count,
    SUM(
        CASE
            WHEN trip_key IS NULL THEN 1
            ELSE 0
        END
    ) AS null_trip_key_count
FROM `ftw-week-08`.`03_gold`.fact_green_taxi_trip;

## Fact Trip Identity Investigation

The five-column combination is treated only as a candidate grouping and is excluded from trip identity and deduplication logic.

The next checks quantify the affected records and compare their measures before any record is removed from the Gold fact table.

In [0]:
WITH candidate_groups AS (
    SELECT
        VendorID,
        lpep_pickup_datetime,
        lpep_dropoff_datetime,
        PULocationID,
        DOLocationID,

        COUNT(*) AS candidate_group_rows,

        COUNT(DISTINCT trip_distance) AS distinct_trip_distances,
        COUNT(DISTINCT passenger_count) AS distinct_passenger_counts,
        COUNT(DISTINCT fare_amount) AS distinct_fare_amounts,
        COUNT(DISTINCT tip_amount) AS distinct_tip_amounts,
        COUNT(DISTINCT total_amount) AS distinct_total_amounts

    FROM `ftw-week-08`.`02_silver`.green_taxi

    GROUP BY
        VendorID,
        lpep_pickup_datetime,
        lpep_dropoff_datetime,
        PULocationID,
        DOLocationID

    HAVING COUNT(*) > 1
)

SELECT
    COUNT(*) AS candidate_group_count,
    SUM(candidate_group_rows) AS candidate_row_count,
    SUM(candidate_group_rows - 1) AS potential_rows_removed_if_deduplicated,

    SUM(
        CASE
            WHEN distinct_trip_distances > 1
              OR distinct_passenger_counts > 1
              OR distinct_fare_amounts > 1
              OR distinct_tip_amounts > 1
              OR distinct_total_amounts > 1
            THEN 1
            ELSE 0
        END
    ) AS groups_with_different_measures

FROM candidate_groups;

## Fact Trip Identity Investigation Interpretation

The five-column combination is only a candidate grouping. Records with different distance, passenger, or financial values remain separate accepted fact rows.

No records are removed by this profiling. Trip deduplication remains controlled by an approved upstream Silver identity rule.

In [0]:
WITH proposed_identity AS (
    SELECT
        xxhash64(
            VendorID,
            lpep_pickup_datetime,
            lpep_dropoff_datetime,
            store_and_fwd_flag,
            RatecodeID,
            PULocationID,
            DOLocationID,
            passenger_count,
            trip_distance,
            trip_duration_minutes,
            fare_amount,
            extra,
            mta_tax,
            tip_amount,
            tolls_amount,
            improvement_surcharge,
            total_amount,
            payment_type,
            trip_type,
            congestion_surcharge,
            cbd_congestion_fee,
            source_file
        ) AS proposed_record_key
    FROM `ftw-week-08`.`02_silver`.green_taxi
)

SELECT
    COUNT(*) AS silver_row_count,
    COUNT(DISTINCT proposed_record_key) AS unique_proposed_record_keys,
    COUNT(*) - COUNT(DISTINCT proposed_record_key)
        AS exact_duplicate_record_count,
    COUNT_IF(proposed_record_key IS NULL)
        AS null_proposed_record_key_count
FROM proposed_identity;

## Deterministic Fact-Row Key Acceptance Criteria

The full-record fingerprint must produce:

- one proposed key for every accepted Silver row
- zero null proposed keys
- zero duplicate proposed keys for the currently accepted data

This is a technical fact-row key, not proof of a unique real-world trip identity. If this validation fails, stop before publishing Gold and resolve identity upstream.

## Validate Fact-to-Dimension Relationships

Confirm that no dimension join multiplies fact rows and that every date, time, taxi-zone, and Weather key matches a dimension member. The documented Unknown Weather member is a valid match.

In [0]:
SELECT
    COUNT(*) AS fact_row_count,
    COUNT(DISTINCT f.trip_key) AS fact_unique_trip_keys,
    COUNT(*) - COUNT(DISTINCT f.trip_key)
        AS duplicate_rows_after_dimension_joins
FROM `ftw-week-08`.`03_gold`.fact_green_taxi_trip AS f
LEFT JOIN `ftw-week-08`.`03_gold`.dim_date AS pickup_date
    ON f.pickup_date_key = pickup_date.date_key
LEFT JOIN `ftw-week-08`.`03_gold`.dim_date AS dropoff_date
    ON f.dropoff_date_key = dropoff_date.date_key
LEFT JOIN `ftw-week-08`.`03_gold`.dim_time AS pickup_time
    ON f.pickup_time_key = pickup_time.time_key
LEFT JOIN `ftw-week-08`.`03_gold`.dim_time AS dropoff_time
    ON f.dropoff_time_key = dropoff_time.time_key
LEFT JOIN `ftw-week-08`.`03_gold`.dim_taxi_zone AS pickup_zone
    ON f.pickup_taxi_zone_key = pickup_zone.taxi_zone_key
LEFT JOIN `ftw-week-08`.`03_gold`.dim_taxi_zone AS dropoff_zone
    ON f.dropoff_taxi_zone_key = dropoff_zone.taxi_zone_key
LEFT JOIN `ftw-week-08`.`03_gold`.dim_weather_hour AS weather
    ON f.pickup_weather_hour_key = weather.weather_hour_key;

## Fact-to-Dimension Join Cardinality Verified Conditions

The joined row count equals the fact-table row count and the duplicate-row count after dimension joins is zero. This confirms that dimension joins do not multiply fact rows.

## Validate FACT Foreign-Key Relationships

The validation checks the pickup/drop-off date, time, and taxi-zone roles plus the pickup Weather-hour relationship. Unavailable Weather is represented by the documented Unknown member instead of a broken foreign key.

In [0]:
SELECT
    SUM(CASE WHEN d_pickup.date_key IS NULL THEN 1 ELSE 0 END)
        AS missing_pickup_date_keys,
    SUM(CASE WHEN d_dropoff.date_key IS NULL THEN 1 ELSE 0 END)
        AS missing_dropoff_date_keys,
    SUM(CASE WHEN t_pickup.time_key IS NULL THEN 1 ELSE 0 END)
        AS missing_pickup_time_keys,
    SUM(CASE WHEN t_dropoff.time_key IS NULL THEN 1 ELSE 0 END)
        AS missing_dropoff_time_keys,
    SUM(CASE WHEN z_pickup.taxi_zone_key IS NULL THEN 1 ELSE 0 END)
        AS missing_pickup_zone_keys,
    SUM(CASE WHEN z_dropoff.taxi_zone_key IS NULL THEN 1 ELSE 0 END)
        AS missing_dropoff_zone_keys,
    SUM(CASE WHEN w.weather_hour_key IS NULL THEN 1 ELSE 0 END)
        AS missing_weather_keys
FROM `ftw-week-08`.`03_gold`.fact_green_taxi_trip AS f
LEFT JOIN `ftw-week-08`.`03_gold`.dim_date AS d_pickup
    ON f.pickup_date_key = d_pickup.date_key
LEFT JOIN `ftw-week-08`.`03_gold`.dim_date AS d_dropoff
    ON f.dropoff_date_key = d_dropoff.date_key
LEFT JOIN `ftw-week-08`.`03_gold`.dim_time AS t_pickup
    ON f.pickup_time_key = t_pickup.time_key
LEFT JOIN `ftw-week-08`.`03_gold`.dim_time AS t_dropoff
    ON f.dropoff_time_key = t_dropoff.time_key
LEFT JOIN `ftw-week-08`.`03_gold`.dim_taxi_zone AS z_pickup
    ON f.pickup_taxi_zone_key = z_pickup.taxi_zone_key
LEFT JOIN `ftw-week-08`.`03_gold`.dim_taxi_zone AS z_dropoff
    ON f.dropoff_taxi_zone_key = z_dropoff.taxi_zone_key
LEFT JOIN `ftw-week-08`.`03_gold`.dim_weather_hour AS w
    ON f.pickup_weather_hour_key = w.weather_hour_key;

## Foreign-Key Verified Conditions

Every missing-key count is zero. Trips without an observed Weather match resolve to the documented Unknown member (`weather_hour_key = 0`).

### Inspect Unexpected Missing Weather References

This query isolates only broken Weather references. Valid Unknown-member assignments are excluded from the result.

In [0]:
SELECT
    f.pickup_weather_hour_key,
    f.pickup_datetime,
    f.dropoff_datetime,
    f.pickup_taxi_zone_key,
    f.dropoff_taxi_zone_key,
    f.source_file,
    f.dq_out_of_range_datetime
FROM `ftw-week-08`.`03_gold`.fact_green_taxi_trip AS f
LEFT JOIN `ftw-week-08`.`03_gold`.dim_weather_hour AS w
    ON f.pickup_weather_hour_key = w.weather_hour_key
WHERE w.weather_hour_key IS NULL
ORDER BY f.pickup_datetime;

## Gold Rerun / Idempotency Signature

The validation signature compares repeated Gold builds against unchanged Silver inputs. Matching row counts, measures, unique-key counts, and the business-row checksum confirm stable rerun behavior. Operational execution timestamps are excluded from the checksum.

In [0]:
SELECT
    COUNT(*) AS fact_row_count,
    COUNT(DISTINCT trip_key) AS unique_trip_keys,
    SUM(trip_count) AS total_trip_count,
    ROUND(SUM(trip_distance), 6) AS total_trip_distance,
    ROUND(SUM(trip_duration_minutes), 6) AS total_trip_duration_minutes,
    ROUND(SUM(total_amount), 6) AS total_amount,
    SUM(CAST(xxhash64(
        trip_key,
        pickup_date_key,
        dropoff_date_key,
        pickup_time_key,
        dropoff_time_key,
        pickup_taxi_zone_key,
        dropoff_taxi_zone_key,
        pickup_weather_hour_key,
        total_amount,
        source_file
    ) AS DECIMAL(38, 0))) AS business_row_checksum
FROM `ftw-week-08`.`03_gold`.fact_green_taxi_trip;

## Analytics Separation

Business-question analysis is intentionally maintained outside this Gold build notebook in three standalone analytics SQL assets:

- `taxi_demand.sql` — BQ1: demand by day, hour, and zone
- `weather_behavior.sql` — BQ2: Weather and trip behavior
- `area_mobility_patterns.sql` — BQ3: pickup/drop-off area patterns and opportunities

This notebook is limited to Gold dimension/fact creation and technical validation. Keeping analytics separate avoids duplicating business logic inside the mart build.

## Final Gold Mart Completion Status

The Gold design, build logic, and validation suite are complete and aligned with the approved Bronze and Silver models.

Gold retains every accepted Silver source record, applies no unapproved Gold-side deduplication, preserves source identifiers and DQ context, and validates row/measure reconciliation, PK/FK relationships, join cardinality, and stable rerun behavior.

Business-question queries are maintained separately in the analytics SQL assets.